<a href="https://colab.research.google.com/github/waqas-manzoor5595/Machine_learning_projects-b-/blob/main/CNN_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Stage 1: Setup

# Task 1 - Import required modules
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.applications import MobileNetV2

# Task 2 - Load the data
base_dir = 'path_to_extracted/weather/'
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')

# Task 3 - Explore the data
train_categories = os.listdir(train_dir)
print("Training categories:", train_categories)

test_categories = os.listdir(test_dir)
print("Testing categories:", test_categories)

# Visualize some sample images
sample_category = train_categories[0]  # Pick one category to explore
sample_img_dir = os.path.join(train_dir, sample_category)
sample_images = os.listdir(sample_img_dir)[:5]  # Select a few images

for img_name in sample_images:
    img_path = os.path.join(sample_img_dir, img_name)
    img = plt.imread(img_path)
    plt.imshow(img)
    plt.title(f"Category: {sample_category}")
    plt.axis('off')
    plt.show()

# Stage 2: Data Preparation

# Task 4 - Resize all images to the same dimensions
img_size = (150, 150)

# Task 5 - Prepare the data for CNN
augmented_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],  # New brightness augmentation
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1.0/255)

train_generator = augmented_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

validation_generator = augmented_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=32,
    class_mode='categorical'
)

# Stage 3: CNN Model

# Task 6 - Define an optimized CNN model
def create_optimized_cnn_model():
    model = Sequential()

    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)))
    model.add(BatchNormalization())  # Add Batch Normalization
    model.add(MaxPooling2D(2, 2))

    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(2, 2))

    model.add(Conv2D(128, (3, 3), activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(2, 2))

    model.add(Flatten())

    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.3))

    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.3))

    model.add(Dense(len(train_categories), activation='softmax'))

    # Compile with a lower learning rate
    model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# Task 7 - Train the CNN Model with Callbacks
optimized_model = create_optimized_cnn_model()
optimized_model.summary()

# Callbacks for better training
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('best_model.h5', save_best_only=True, monitor='val_accuracy')
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)

# Train the model
optimized_history = optimized_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=20,
    callbacks=[early_stopping, model_checkpoint, lr_scheduler]
)

# Stage 4: Testing

# Task 8 - Evaluate on Test Data
test_loss, test_acc = optimized_model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc * 100:.2f}%")

# Stage 5: Transfer Learning (Optional)
def create_transfer_learning_model():
    base_model = MobileNetV2(input_shape=(150, 150, 3), include_top=False, weights='imagenet')
    base_model.trainable = False  # Freeze base model layers

    model = Sequential([
        base_model,
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(len(train_categories), activation='softmax')
    ])

    model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# Uncomment to train MobileNetV2-based model
# transfer_model = create_transfer_learning_model()
# transfer_model.summary()
# transfer_history = transfer_model.fit(
#     train_generator,
#     validation_data=validation_generator,
#     epochs=15,
#     callbacks=[early_stopping, model_checkpoint, lr_scheduler]
# )


FileNotFoundError: [Errno 2] No such file or directory: 'path_to_extracted/weather/train'